# Milestone 0: EDA and Data Preparation

This notebook prepares the first reproducible dataset sample for the Smart Product Intelligence capstone project.

**Dataset:** `McAuley-Lab/Amazon-Reviews-2023`  
**Category:** `All_Beauty`  
**Review config:** `raw_review_All_Beauty`  
**Metadata config:** `raw_meta_All_Beauty`

The goal for Milestone 0 is data loading, cleaning, joining, exploratory analysis, and leakage-safe train/validation/test splitting. Model development starts later.

## 1. Setup

We keep the sample size manageable for local experimentation: at most 30,000 reviews and 10,000 product metadata rows. The notebook writes processed split files to `data/processed/`.

In [ ]:
from __future__ import annotations

DATASET_NAME = "McAuley-Lab/Amazon-Reviews-2023"
REVIEW_CONFIG = "raw_review_All_Beauty"
META_CONFIG = "raw_meta_All_Beauty"
MAX_REVIEWS = 30000
MAX_PRODUCTS = 10000
RANDOM_SEED = 42

import json
import re
import warnings
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {PROCESSED_DIR}")


## 2. Load Hugging Face Subsets

The Amazon Reviews 2023 dataset is large, so this notebook uses streaming first and only materializes the bounded subset needed for Milestone 0. If streaming is unavailable in a local environment, the helper falls back to regular `datasets` loading and selects the first `max_rows` records.

In [ ]:
def load_hf_subset(config_name: str, max_rows: int) -> pd.DataFrame:
    """Load a bounded subset from a Hugging Face dataset config."""

    print(f"Loading up to {max_rows:,} rows from {config_name}...")
    try:
        stream = load_dataset(
            DATASET_NAME,
            config_name,
            split="full",
            streaming=True,
            trust_remote_code=True,
        )
        records = list(islice(stream, max_rows))
        df = pd.DataFrame.from_records(records)
    except Exception as exc:
        print(f"Streaming failed for {config_name}: {exc}")
        print("Falling back to regular Hugging Face loading.")
        dataset = load_dataset(
            DATASET_NAME,
            config_name,
            split="full",
            trust_remote_code=True,
        )
        row_count = min(max_rows, len(dataset))
        df = dataset.select(range(row_count)).to_pandas()

    print(f"Loaded shape for {config_name}: {df.shape}")
    return df


reviews_raw = load_hf_subset(REVIEW_CONFIG, MAX_REVIEWS)
metadata_raw = load_hf_subset(META_CONFIG, MAX_PRODUCTS)

## 3. Inspect Raw Columns

Before cleaning, inspect the raw schemas. The expected product-level join key is `parent_asin`, but the cleaning code below checks for fallback ID columns to avoid fragile hard-coded assumptions.

In [ ]:
print("Reviews shape:", reviews_raw.shape)
print("Review columns:")
display(pd.DataFrame({"column": reviews_raw.columns.tolist()}))
display(reviews_raw.head())

print("Metadata shape:", metadata_raw.shape)
print("Metadata columns:")
display(pd.DataFrame({"column": metadata_raw.columns.tolist()}))
display(metadata_raw.head())

## 4. Cleaning Helpers

These helper functions keep the notebook robust when optional columns are missing, prices are stored as strings, or image fields contain nested lists/dictionaries with missing links.

In [ ]:
PRODUCT_ID_CANDIDATES = ["parent_asin", "asin", "product_id", "item_id"]
TEXT_CANDIDATES = ["text", "review_text", "review_body"]
REVIEW_TITLE_CANDIDATES = ["title", "review_title", "summary"]
RATING_CANDIDATES = ["rating", "overall", "score"]
IMAGE_CANDIDATES = ["images", "image", "image_url", "image_urls"]


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """Return the first available column from a candidate list."""

    return next((column for column in candidates if column in df.columns), None)


def series_or_default(df: pd.DataFrame, column: str | None, default_value=pd.NA) -> pd.Series:
    """Return a DataFrame column or a same-length default Series."""

    if column and column in df.columns:
        return df[column]
    return pd.Series([default_value] * len(df), index=df.index)


def clean_product_id(series: pd.Series) -> pd.Series:
    """Normalize product IDs while preserving missing values."""

    cleaned = series.astype("string").str.strip()
    return cleaned.mask(cleaned.str.lower().isin(["", "none", "nan", "null", "<na>"]))


def clean_text(series: pd.Series, fill_value: str = "") -> pd.Series:
    """Normalize text columns without failing on missing values."""

    return series.fillna(fill_value).astype(str).str.strip()


def is_null_like(value) -> bool:
    """Safely detect scalar null-like values without breaking nested objects."""

    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    if isinstance(value, str):
        return value.strip().lower() in {"", "none", "nan", "null", "n/a", "[]", "{}"}
    return False


def extract_price(value) -> float:
    """Extract a numeric price from strings like '$12.99' or return NaN."""

    if is_null_like(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).replace(",", "").strip()
    match = re.search(r"\d+(?:\.\d+)?", text)
    if not match:
        return np.nan

    try:
        return float(match.group(0))
    except ValueError:
        return np.nan


def has_any_image(value) -> bool:
    """Detect whether a nested image field contains at least one URL-like value."""

    if is_null_like(value):
        return False

    if isinstance(value, str):
        stripped = value.strip()
        if stripped.startswith(("http://", "https://")):
            return True
        try:
            return has_any_image(json.loads(stripped))
        except Exception:
            return False

    if isinstance(value, dict):
        return any(has_any_image(item) for item in value.values())

    if isinstance(value, (list, tuple, set)):
        return any(has_any_image(item) for item in value)

    return False


def missing_values_report(df: pd.DataFrame) -> pd.DataFrame:
    """Create a sorted missing-value report for a DataFrame."""

    report = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_percent": (df.isna().mean().values * 100).round(2),
        }
    )
    return report.sort_values(["missing_percent", "missing_count"], ascending=False)

## 5. Clean Reviews and Product Metadata

The cleaned tables standardize product IDs, review text, ratings, review length, price, and image availability flags. Optional fields are added only when available.

In [ ]:
review_product_col = find_column(reviews_raw, PRODUCT_ID_CANDIDATES)
metadata_product_col = find_column(metadata_raw, PRODUCT_ID_CANDIDATES)

if review_product_col is None:
    raise ValueError(f"Could not find a review product ID column. Available columns: {reviews_raw.columns.tolist()}")
if metadata_product_col is None:
    raise ValueError(f"Could not find a metadata product ID column. Available columns: {metadata_raw.columns.tolist()}")

review_text_col = find_column(reviews_raw, TEXT_CANDIDATES)
review_title_col = find_column(reviews_raw, REVIEW_TITLE_CANDIDATES)
review_rating_col = find_column(reviews_raw, RATING_CANDIDATES)
review_image_col = find_column(reviews_raw, IMAGE_CANDIDATES)

metadata_title_col = "title" if "title" in metadata_raw.columns else None
metadata_image_col = find_column(metadata_raw, IMAGE_CANDIDATES)

print(f"Review product ID column: {review_product_col}")
print(f"Metadata product ID column: {metadata_product_col}")

reviews = pd.DataFrame(index=reviews_raw.index)
reviews["product_id"] = clean_product_id(series_or_default(reviews_raw, review_product_col))
reviews["review_rating"] = pd.to_numeric(series_or_default(reviews_raw, review_rating_col, np.nan), errors="coerce")
reviews["review_title"] = clean_text(series_or_default(reviews_raw, review_title_col, ""))
reviews["review_text"] = clean_text(series_or_default(reviews_raw, review_text_col, ""))
reviews["review_length"] = reviews["review_text"].str.split().str.len().fillna(0).astype(int)
reviews["review_images_raw"] = series_or_default(reviews_raw, review_image_col, None)
reviews["has_review_image"] = reviews["review_images_raw"].apply(has_any_image)

for optional_column in ["asin", "user_id", "timestamp", "helpful_vote", "verified_purchase"]:
    if optional_column in reviews_raw.columns:
        reviews[optional_column] = reviews_raw[optional_column]

if "timestamp" in reviews.columns:
    reviews["review_datetime"] = pd.to_datetime(reviews["timestamp"], unit="ms", errors="coerce")

reviews = reviews.dropna(subset=["product_id"]).head(MAX_REVIEWS)
selected_product_ids = reviews["product_id"].drop_duplicates().head(MAX_PRODUCTS)
reviews = reviews[reviews["product_id"].isin(selected_product_ids)].reset_index(drop=True)

metadata = pd.DataFrame(index=metadata_raw.index)
metadata["product_id"] = clean_product_id(series_or_default(metadata_raw, metadata_product_col))
metadata["product_title"] = clean_text(series_or_default(metadata_raw, metadata_title_col, "Unknown product"), "Unknown product")
metadata["main_category"] = clean_text(series_or_default(metadata_raw, "main_category", "Unknown"), "Unknown")
metadata["store"] = clean_text(series_or_default(metadata_raw, "store", "Unknown"), "Unknown")
metadata["average_rating"] = pd.to_numeric(series_or_default(metadata_raw, "average_rating", np.nan), errors="coerce")
metadata["rating_number"] = pd.to_numeric(series_or_default(metadata_raw, "rating_number", np.nan), errors="coerce")
metadata["price_raw"] = series_or_default(metadata_raw, "price", np.nan)
metadata["price_numeric"] = metadata["price_raw"].apply(extract_price)
metadata["product_images_raw"] = series_or_default(metadata_raw, metadata_image_col, None)
metadata["has_product_image"] = metadata["product_images_raw"].apply(has_any_image)

for optional_column in ["features", "description", "categories", "details", "videos", "bought_together", "subtitle", "author"]:
    if optional_column in metadata_raw.columns:
        metadata[f"product_{optional_column}"] = metadata_raw[optional_column]

metadata = (
    metadata.dropna(subset=["product_id"])
    .drop_duplicates(subset=["product_id"])
    .head(MAX_PRODUCTS)
    .reset_index(drop=True)
)

print("Cleaned reviews shape:", reviews.shape)
print("Cleaned metadata shape:", metadata.shape)
display(reviews.head())
display(metadata.head())

## 6. Missing Values Before Join

Missingness is expected in optional metadata fields such as price, images, subtitles, and author. We report missing values before and after joining so later milestones can make explicit feature decisions.

In [ ]:
print("Reviews missing values:")
display(missing_values_report(reviews))

print("Metadata missing values:")
display(missing_values_report(metadata))

## 7. Join Reviews With Product Metadata

The join uses the product-level ID column. For this dataset, `parent_asin` is the expected shared key. We use a left join from reviews to metadata to preserve the sampled reviews while keeping metadata fields where available.

In [ ]:
merged = reviews.merge(metadata, on="product_id", how="left", validate="many_to_one")

for text_column in ["review_title", "review_text", "product_title", "main_category", "store"]:
    if text_column in merged.columns:
        fill_value = "Unknown" if text_column in {"product_title", "main_category", "store"} else ""
        merged[text_column] = merged[text_column].fillna(fill_value)

for flag_column in ["has_review_image", "has_product_image"]:
    if flag_column in merged.columns:
        merged[flag_column] = merged[flag_column].fillna(False).astype(bool)

joined_metadata_rate = merged["product_title"].ne("Unknown").mean() * 100 if "product_title" in merged else 0
print("Merged shape:", merged.shape)
print(f"Rows with matched metadata: {joined_metadata_rate:.2f}%")
display(merged.head())
display(missing_values_report(merged).head(20))

## 8. EDA Visualizations

These plots cover the required Milestone 0 checks: rating distribution, price distribution, review length distribution, missing value report, and image availability counts.

In [ ]:
def no_data_message(ax, title: str, message: str = "No usable data") -> None:
    ax.set_title(title)
    ax.text(0.5, 0.5, message, ha="center", va="center")
    ax.set_axis_off()


fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

rating_data = merged["review_rating"].dropna() if "review_rating" in merged else pd.Series(dtype=float)
if not rating_data.empty:
    sns.countplot(x=rating_data, ax=axes[0], color="#4C78A8")
    axes[0].set_title("Rating Distribution")
    axes[0].set_xlabel("Review rating")
    axes[0].set_ylabel("Count")
else:
    no_data_message(axes[0], "Rating Distribution")

price_data = merged["price_numeric"].dropna() if "price_numeric" in merged else pd.Series(dtype=float)
if not price_data.empty:
    upper = price_data.quantile(0.99)
    clipped_price_data = price_data[price_data <= upper]
    sns.histplot(clipped_price_data, bins=40, kde=True, ax=axes[1], color="#59A14F")
    axes[1].set_title("Price Distribution (clipped at 99th percentile)")
    axes[1].set_xlabel("Price")
else:
    no_data_message(axes[1], "Price Distribution", "No numeric prices extracted")

length_data = merged["review_length"].dropna() if "review_length" in merged else pd.Series(dtype=float)
if not length_data.empty:
    upper = max(1, length_data.quantile(0.99))
    clipped_length_data = length_data[length_data <= upper]
    sns.histplot(clipped_length_data, bins=40, kde=True, ax=axes[2], color="#F28E2B")
    axes[2].set_title("Review Length Distribution (words, clipped at 99th percentile)")
    axes[2].set_xlabel("Review length")
else:
    no_data_message(axes[2], "Review Length Distribution")

missing_report = missing_values_report(merged)
missing_plot = missing_report[missing_report["missing_count"] > 0].head(15)
if not missing_plot.empty:
    sns.barplot(data=missing_plot, y="column", x="missing_percent", ax=axes[3], color="#E15759")
    axes[3].set_title("Top Missing Values")
    axes[3].set_xlabel("Missing percent")
    axes[3].set_ylabel("")
else:
    no_data_message(axes[3], "Missing Values", "No missing values detected")

image_rows = []
for column, label in [("has_product_image", "Product metadata"), ("has_review_image", "Review content")]:
    if column in merged.columns:
        counts = merged[column].value_counts(dropna=False)
        for has_image, count in counts.items():
            image_rows.append({"source": label, "has_image": bool(has_image), "count": int(count)})

image_counts = pd.DataFrame(image_rows)
if not image_counts.empty:
    sns.barplot(data=image_counts, x="source", y="count", hue="has_image", ax=axes[4], palette=["#B07AA1", "#76B7B2"])
    axes[4].set_title("Image Availability Count")
    axes[4].set_xlabel("")
    axes[4].set_ylabel("Count")
    axes[4].legend(title="Has image")
else:
    no_data_message(axes[4], "Image Availability Count")

axes[5].axis("off")
plt.tight_layout()
plt.show()

display(missing_report)

## 9. Split by Product to Avoid Leakage

The split is performed on unique product IDs, not on individual reviews. This prevents reviews for the same product from appearing across train, validation, and test sets.

In [ ]:
def split_product_ids(product_ids: pd.Series, random_state: int = RANDOM_SEED) -> tuple[set[str], set[str], set[str]]:
    """Split product IDs into 70/15/15 train/validation/test sets."""

    unique_products = product_ids.dropna().drop_duplicates().astype(str)
    product_count = len(unique_products)
    if product_count < 3:
        raise ValueError("Need at least 3 unique products to create train/validation/test splits.")

    if product_count < 10:
        shuffled = unique_products.sample(frac=1, random_state=random_state).tolist()
        train_count = max(1, int(round(product_count * 0.70)))
        validation_count = max(1, int(round(product_count * 0.15)))
        if train_count + validation_count >= product_count:
            train_count = product_count - 2
            validation_count = 1

        train_ids = set(shuffled[:train_count])
        validation_ids = set(shuffled[train_count : train_count + validation_count])
        test_ids = set(shuffled[train_count + validation_count :])
        return train_ids, validation_ids, test_ids

    train_ids, temp_ids = train_test_split(
        unique_products,
        test_size=0.30,
        random_state=random_state,
        shuffle=True,
    )
    validation_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.50,
        random_state=random_state,
        shuffle=True,
    )
    return set(train_ids), set(validation_ids), set(test_ids)


split_ready = merged.dropna(subset=["product_id"]).copy()
train_product_ids, validation_product_ids, test_product_ids = split_product_ids(split_ready["product_id"])

train_df = split_ready[split_ready["product_id"].isin(train_product_ids)].copy()
validation_df = split_ready[split_ready["product_id"].isin(validation_product_ids)].copy()
test_df = split_ready[split_ready["product_id"].isin(test_product_ids)].copy()

assert train_product_ids.isdisjoint(validation_product_ids)
assert train_product_ids.isdisjoint(test_product_ids)
assert validation_product_ids.isdisjoint(test_product_ids)

split_summary = pd.DataFrame(
    [
        {"split": "train", "reviews": len(train_df), "products": train_df["product_id"].nunique()},
        {"split": "validation", "reviews": len(validation_df), "products": validation_df["product_id"].nunique()},
        {"split": "test", "reviews": len(test_df), "products": test_df["product_id"].nunique()},
    ]
)
split_summary["review_percent"] = (split_summary["reviews"] / split_summary["reviews"].sum() * 100).round(2)
split_summary["product_percent"] = (split_summary["products"] / split_summary["products"].sum() * 100).round(2)
display(split_summary)

## 10. Save Processed Splits

The processed files are saved as CSVs under `data/processed/`. Nested list/dictionary columns are serialized to JSON strings so the output is CSV-friendly.

In [ ]:
def serialize_nested_value(value):
    """Serialize nested values for CSV output."""

    if isinstance(value, (dict, list, tuple, set)):
        try:
            return json.dumps(value, ensure_ascii=False)
        except TypeError:
            return str(value)
    return value


def make_csv_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Convert nested object columns to CSV-safe strings."""

    csv_df = df.copy()
    object_columns = csv_df.select_dtypes(include=["object"]).columns
    for column in object_columns:
        if csv_df[column].map(lambda value: isinstance(value, (dict, list, tuple, set))).any():
            csv_df[column] = csv_df[column].map(serialize_nested_value)
    return csv_df


split_files = {
    "train": PROCESSED_DIR / "train.csv",
    "validation": PROCESSED_DIR / "validation.csv",
    "test": PROCESSED_DIR / "test.csv",
}

make_csv_safe(train_df).to_csv(split_files["train"], index=False)
make_csv_safe(validation_df).to_csv(split_files["validation"], index=False)
make_csv_safe(test_df).to_csv(split_files["test"], index=False)

for split_name, path in split_files.items():
    print(f"Saved {split_name}: {path}")

## Milestone 0 Complete

At this point we have:

- Loaded bounded review and metadata subsets from Hugging Face.
- Cleaned review text, review length, product metadata, price, and image availability fields.
- Joined reviews to product metadata by product ID.
- Created EDA plots for ratings, prices, review length, missing values, and images.
- Split data by product ID into train/validation/test sets to reduce leakage risk.
- Saved CSV splits to `data/processed/`.

Do not start Milestone 1 modeling work in this notebook.